In [1]:
import pandas as pd

# Agar files direct folder mein hain (no subfolder)
deliveries_path = './IPL Datasets/deliveries.csv'   # ya pura path daal de jaise '/content/ipl dataset/deliveries.csv' Colab mein
matches_path = './IPL Datasets/matches.csv'

# Load kar
deliveries = pd.read_csv(deliveries_path)
matches = pd.read_csv(matches_path)

# Columns dekh
print("Deliveries.csv ke columns:")
print(deliveries.columns.tolist())

print("\nMatches.csv ke columns:")
print(matches.columns.tolist())

# Size bhi check kar
print("\nDeliveries shape (rows, columns):", deliveries.shape)
print("Matches shape:", matches.shape)

# Top 5 rows dekhne ke liye (optional)
print(deliveries.head())
print(matches.head())

Deliveries.csv ke columns:
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

Matches.csv ke columns:
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

Deliveries shape (rows, columns): (260920, 17)
Matches shape: (1095, 20)
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4  

In [2]:
# Sirf zaroori columns rakh deliveries se
deliveries = deliveries[['match_id', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
                         'batter', 'batsman_runs', 'extra_runs', 'total_runs',
                         'is_wicket', 'player_dismissed']]

# Extra check (already hai to safe)
print("Selected deliveries columns:", deliveries.columns.tolist())

Selected deliveries columns: ['match_id', 'inning', 'over', 'ball', 'batting_team', 'bowling_team', 'batter', 'batsman_runs', 'extra_runs', 'total_runs', 'is_wicket', 'player_dismissed']


In [3]:
# Group by match_id + inning
deliveries['current_runs'] = deliveries.groupby(['match_id', 'inning'])['total_runs'].cumsum()

# Current wickets (is_wicket use kar rahe hain)
deliveries['current_wickets'] = deliveries.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

# Balls played calculate (over aur ball se)
deliveries['over_number'] = deliveries['over'].astype(int)          # 0,1,2,...
deliveries['ball_in_over'] = deliveries['ball'].astype(int)         # 1 to 6 usually
deliveries['balls_played'] = (deliveries['over_number'] * 6) + deliveries['ball_in_over']

# Balls remaining (T20 = 120 balls)
deliveries['balls_remaining'] = 120 - deliveries['balls_played']

# Current run rate
deliveries['current_run_rate'] = deliveries['current_runs'] / (deliveries['balls_played'] / 6.0).replace(0, 1)  # avoid div by 0

# Sample check kar
print(deliveries[['match_id', 'inning', 'over', 'ball', 'current_runs', 'current_wickets', 
                  'balls_played', 'balls_remaining', 'current_run_rate']].head(15))

    match_id  inning  over  ball  current_runs  current_wickets  balls_played  \
0     335982       1     0     1             1                0             1   
1     335982       1     0     2             1                0             2   
2     335982       1     0     3             2                0             3   
3     335982       1     0     4             2                0             4   
4     335982       1     0     5             2                0             5   
5     335982       1     0     6             2                0             6   
6     335982       1     0     7             3                0             7   
7     335982       1     1     1             3                0             7   
8     335982       1     1     2             7                0             8   
9     335982       1     1     3            11                0             9   
10    335982       1     1     4            17                0            10   
11    335982       1     1  

In [4]:
# Har innings ka final score = max current_runs us innings mein
final_scores = deliveries.groupby(['match_id', 'inning'])['current_runs'].max().reset_index()
final_scores = final_scores.rename(columns={'current_runs': 'final_score'})

# Merge back
deliveries = deliveries.merge(final_scores, on=['match_id', 'inning'], how='left')

print("Sample rows with final_score:")
print(deliveries[['match_id', 'inning', 'over', 'current_runs', 'final_score']].sample(5))

Sample rows with final_score:
        match_id  inning  over  current_runs  final_score
256472   1426291       1     0             6          167
131492    980975       2    14           135          164
55115     501256       1    16            97          135
78520     598006       2    15           132          165
80449     598014       2    15            95          126


In [5]:
# Sirf first innings (batting first wali)
df = deliveries[deliveries['inning'] == 1].copy()

# Optional: Pehle 5 overs ke baad ke data use karo (early prediction ke liye better)
# Agar full innings chahiye to yeh line comment kar de
df = df[df['over_number'] >= 5]

print("Filtered df shape:", df.shape)

Filtered df shape: (100841, 20)


In [6]:
# Basic features jo model mein jayenge
features = ['batting_team', 'bowling_team', 'current_runs', 'current_wickets',
            'balls_remaining', 'current_run_rate', 'over_number']

# One-hot encoding teams ke liye
df_model = pd.get_dummies(df[features + ['final_score']], 
                          columns=['batting_team', 'bowling_team'],
                          drop_first=True)   # avoid dummy trap

print("Ready for modeling - columns:")
print(df_model.columns.tolist())

print("\nFirst 3 rows sample:")
print(df_model.head(3))

Ready for modeling - columns:
['current_runs', 'current_wickets', 'balls_remaining', 'current_run_rate', 'over_number', 'final_score', 'batting_team_Deccan Chargers', 'batting_team_Delhi Capitals', 'batting_team_Delhi Daredevils', 'batting_team_Gujarat Lions', 'batting_team_Gujarat Titans', 'batting_team_Kings XI Punjab', 'batting_team_Kochi Tuskers Kerala', 'batting_team_Kolkata Knight Riders', 'batting_team_Lucknow Super Giants', 'batting_team_Mumbai Indians', 'batting_team_Pune Warriors', 'batting_team_Punjab Kings', 'batting_team_Rajasthan Royals', 'batting_team_Rising Pune Supergiant', 'batting_team_Rising Pune Supergiants', 'batting_team_Royal Challengers Bangalore', 'batting_team_Royal Challengers Bengaluru', 'batting_team_Sunrisers Hyderabad', 'bowling_team_Deccan Chargers', 'bowling_team_Delhi Capitals', 'bowling_team_Delhi Daredevils', 'bowling_team_Gujarat Lions', 'bowling_team_Gujarat Titans', 'bowling_team_Kings XI Punjab', 'bowling_team_Kochi Tuskers Kerala', 'bowling_tea

In [13]:
!pip install xgboost --no-cache-dir

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 3.4/101.7 MB 18.3 MB/s eta 0:00:06
   -- ------------------------------------- 6.3/101.7 MB 16.7 MB/s eta 0:00:06
   --- ------------------------------------ 9.2/101.7 MB 15.4 MB/s eta 0:00:06
   ---- ----------------------------------- 11.5/101.7 MB 14.7 MB/s eta 0:00:07
   ----- ---------------------------------- 14.4/101.7 MB 14.6 MB/s eta 0:00:06
   ------ --------------------------------- 17.3/101.7 MB 14.2 MB/s eta 0:00:06
   ------- -------------------------------- 20.2/101.7 MB 14.0 MB/s eta 0:00:06
   -------- ------------------------------- 22.8/101.7 MB 13.9 MB/s eta 0:00:06
   ---------- ----------------------------- 25.4/101.7 MB 13.9 MB/s eta 0:00:06
   ----------- ---------------------------- 28.0/101.7 MB 13.8 MB/s eta 0:00:06
   ----------- ---------------------------- 30.4/101.7 MB 13.8 MB/s eta 0:00:06
   ----------- ---------------------------- 30.4/101

In [14]:
import xgboost
print(xgboost.__version__)

3.2.0


In [18]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# X aur y banao
X = df_model.drop('final_score', axis=1)
y = df_model['final_score']

# Split (80% train, 20% test) – random_state fix kar denge reproducibility ke liye
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# XGBoost model
model = XGBRegressor(
    n_estimators=500,      # trees kitne
    learning_rate=0.05,    # slow learning better accuracy
    max_depth=6,           # tree depth
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1              # fast training
)

# Train kar
model.fit(X_train, y_train)

# Predict on test
y_pred = model.predict(X_test)

# Errors check kar
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Mean Absolute Error (MAE): {mae:.2f} runs")          # Goal: <12-15 runs achha hai
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} runs")

# Matches ke 'id' column ko 'match_id' naam de denge taaki merge easy ho
matches = matches.rename(columns={'id': 'match_id'})

# df (jo pehle bana tha first innings wala) mein venue add kar do
df = df.merge(matches[['match_id', 'venue']], on='match_id', how='left')

# Check kar le venue add hua ya nahi
print("Venue added? Sample:")
print(df[['match_id', 'venue', 'over', 'current_runs']].sample(5))

print("\nKitne rows mein venue missing hai?", df['venue'].isnull().sum())

Train shape: (80672, 41)
Test shape: (20169, 41)
Mean Absolute Error (MAE): 9.72 runs
Root Mean Squared Error (RMSE): 13.56 runs
Venue added? Sample:
       match_id                                              venue  over  \
93296   1359539  Rajiv Gandhi International Stadium, Uppal, Hyd...    17   
30274    598007         Punjab Cricket Association Stadium, Mohali     8   
17759    501218                                   Wankhede Stadium    10   
27463    548360                                   Wankhede Stadium    19   
79396   1254107                            Sharjah Cricket Stadium    18   

       current_runs  
93296           161  
30274            61  
17759            67  
27463           126  
79396           109  

Kitne rows mein venue missing hai? 0


In [19]:
# Features list mein venue add kar
features_with_venue = features + ['venue']   # features pehle se defined hai jaise ['batting_team', 'bowling_team', 'current_runs', ...]

# One-hot encoding (teams + venue sab ke liye)
df_model_venue = pd.get_dummies(
    df[features_with_venue + ['final_score']],
    columns=['batting_team', 'bowling_team', 'venue'],
    drop_first=True   # extra columns avoid karne ke liye
)

print("Naye columns count:", len(df_model_venue.columns))
print("Shape:", df_model_venue.shape)
print("Sample columns:", df_model_venue.columns[-10:].tolist())  # last 10 columns dekhne ke liye (venue wale)

Naye columns count: 99
Shape: (100841, 99)
Sample columns: ['venue_Shaheed Veer Narayan Singh International Stadium', 'venue_Sharjah Cricket Stadium', 'venue_Sheikh Zayed Stadium', "venue_St George's Park", 'venue_Subrata Roy Sahara Stadium', 'venue_SuperSport Park', 'venue_Vidarbha Cricket Association Stadium, Jamtha', 'venue_Wankhede Stadium', 'venue_Wankhede Stadium, Mumbai', 'venue_Zayed Cricket Stadium, Abu Dhabi']


In [20]:
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# X aur y
X = df_model_venue.drop('final_score', axis=1)
y = df_model_venue['final_score']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model (same as before)
model_venue = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_venue.fit(X_train, y_train)

# Predict
y_pred = model_venue.predict(X_test)

# Results
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE (with venue): {mae:.2f} runs")
print(f"RMSE (with venue): {rmse:.2f} runs")

MAE (with venue): 8.34 runs
RMSE (with venue): 11.51 runs


In [21]:
import joblib

# Model save
joblib.dump(model_venue, 'ipl_score_predictor_model.pkl')

# Ya XGBoost native format
model_venue.save_model('ipl_score_predictor_xgb.json')

print("Model saved!")

Model saved!


In [22]:
# Ek sample input bana (example: MI batting first at Wankhede)
sample_data = {
    'current_runs': 95,
    'current_wickets': 3,
    'balls_remaining': 54,
    'current_run_rate': 7.92,
    'over_number': 11,
    # One-hot columns manually set (sirf relevant 1 kar, baaki 0)
    # Batting team example: Mumbai Indians
    'batting_team_Mumbai Indians': 1,
    'batting_team_Chennai Super Kings': 0,
    # ... baaki batting teams 0
    # Bowling team example: Royal Challengers Bengaluru
    'bowling_team_Royal Challengers Bengaluru': 1,
    # ... baaki 0
    # Venue example: Wankhede Stadium
    'venue_Wankhede Stadium': 1,
    # ... baaki venues 0
}

# DataFrame bana
import pandas as pd
sample_df = pd.DataFrame([sample_data])

# Missing columns ko 0 se fill (agar kuch one-hot miss hua)
sample_df = sample_df.reindex(columns=X.columns, fill_value=0)

predicted = model_venue.predict(sample_df)[0]
print(f"Predicted final score: {predicted:.0f} runs (±8 runs error ke saath)")

Predicted final score: 180 runs (±8 runs error ke saath)
